# Model Deployment

This notebook is for running the deployed model server without running all experiments from beginning

### Import All Dependency

In [1]:
import os 
import re
import torch
import threading
from flask import Flask, jsonify, request
from werkzeug.serving import make_server
from pyngrok import ngrok
from dotenv import load_dotenv
from transformers import BartTokenizer, BartForConditionalGeneration

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
MODEL_DIR = os.path.join(BASE_DIR, "models", "phase2", "bart_model")


In [2]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

## Load the Model

In [3]:
# Helper
def build_prompt(review_text):
    return (
        "### Instruction: Extract the key aspects and their sentiments from the following game review. "
        "You MUST format the output strictly as Aspects: "
        "[category:aspect:sentiment, ...]\n\n"
        f"### Input: Review: \"{review_text}\"\n\n"
        "### Response:"
    )
    
def parse_predicted_aspects(pred_text):
    cleaned = re.sub(r"Aspects\s*:\s*\[?", "", pred_text, flags=re.IGNORECASE).replace("]", "")
    matches = re.findall(
        r"([A-Za-z0-9_/]+)\s*:\s*([^:\],]+?)\s*:\s*(positive|negative|neutral)",
        cleaned,
        flags=re.IGNORECASE
    )
    return [
        {"category": cat.strip(), "aspect": asp.strip(), "sentiment": sent.strip().capitalize()}
        for cat, asp, sent in matches
    ]

In [4]:
bart_tokenizer = BartTokenizer.from_pretrained(MODEL_DIR)
deploy_model = BartForConditionalGeneration.from_pretrained(MODEL_DIR).to(device)
deploy_model.eval()

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (fi

## Setup The Deployment

In [5]:
load_dotenv(os.path.join(BASE_DIR, ".env"))
api_key = os.getenv("NGROK_PHASE_2_TOKEN")

if api_key:
    try:
        ngrok.set_auth_token(api_key)
    except Exception:
        pass

In [6]:
app = Flask(__name__)
port = 5001
public_url = ngrok.connect(port).public_url
print(f' * Ngrok tunnel LIVE at: "{public_url}" -> "http://127.0.0.1:{port}"')

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "healthy",
        "phase": "Phase 2 - ABSA Tagger",
        "model": "facebook/bart-base (Task-Adapted)",
        "device": str(device),
        "model_loaded": deploy_model is not None
    }), 200

@app.route("/inference", methods=["GET", "POST"])
def inference():
    if request.method == "GET":
        app_name = request.args.get("app_name", "Unknown Game")
        rev = request.args.get("review") or request.args.get("content")
        if not rev:
            return jsonify({
                "status": "Hello, I'm Phase 2 ABSA!",
                "usage_post": "POST JSON {'app_name': '...', 'reviews': ['...']}",
                "usage_get": "GET /inference?app_name=Counter-Strike&review=Great+gameplay+and+music"
            })
        reviews = [rev]
      
    elif request.method == "POST":
        if not request.is_json:
            return jsonify({"msg": "POST body must be JSON"}), 400
        
        body = request.json
        app_name = body.get("app_name", "Unknown Game")
        reviews = body.get("reviews") or body.get("review") or body.get("content", [])
        
        if isinstance(reviews, str):
            reviews = [reviews]
            
        prompts = [build_prompt(review) for review in reviews]
        inputs = bart_tokenizer(prompts, return_tensors="pt", padding=True, max_length=1024, truncation=True).to(device)
        
        with torch.no_grad():
            out_ids = deploy_model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                repetition_penalty=1.0,
                length_penalty=1.5,
                early_stopping=True
            )
            
        raw_preds = bart_tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        all_aspects = []
        seen = set()
        
        for review, pred_text in zip(reviews, raw_preds):
            for item in parse_predicted_aspects(pred_text):
                if item["aspect"].lower() not in review.lower():
                    continue
                key = (item["category"].lower(), item["aspect"].lower())
                if key not in seen:
                    seen.add(key)
                    all_aspects.append(item)
                    
        return jsonify({
            "app_name": app_name,
            "reviews": reviews,
            "aspects": all_aspects,
            "raw_prediction": raw_preds
        })

 * Ngrok tunnel LIVE at: "https://swirl-abruptly-outclass.ngrok-free.dev" -> "http://127.0.0.1:5001"


In [7]:
class ThreadingServer(threading.Thread):
  def __init__(self, app, port):
    super().__init__()
    self.server = make_server("127.0.0.1", port, app)

  def run(self):
    print("Starting server")
    self.server.serve_forever()

  def shutdown(self):
    print("Shutting down server")
    self.server.shutdown()

In [8]:
print(f' * ngrok tunnel "{public_url}" -> "http://127.0.0.1:{port}"')
server_thread = ThreadingServer(app, port)
server_thread.start()

 * ngrok tunnel "https://swirl-abruptly-outclass.ngrok-free.dev" -> "http://127.0.0.1:5001"
Starting server


In [9]:
import requests
url = "https://swirl-abruptly-outclass.ngrok-free.dev/inference"

payload = {
        "app_name": "Redemption: Eternal Quest",
        "reviews": [
            "The interface needs a bit more polishing, and i was pretty confused at the initial team set up, I think a tutorial would be more friendly. Took me a while to really figure out what was going on, then i started enjoying the game, it's quite casual, and the backgound music really compliment the game a lot. For like only three bucks i couldn't give it a thumb down.",
            "Played the game for about 30 minutes. I am not really impressed, it is missing the wow factor that would make me want to continue playing any longer. *During my play through I place my adventurers on many quests and was unable to raise my gold level past 10,000. It was a continual loss of gold, with very little gains. *You contract your adventurers for 1-4 years. Each quest is a season long, four quests equal 1 year. At the end of their contract you have to pay for them again or contract a fresh adventure. So if I can't make money and have to pay large amounts of gold for unskilled adventures what the hell is the point. *If there was a way to allow my adventures to rest without having to send anyone else out on a quest, it was lost in the explaination on how to play. *Can't equip your adventurers to make them better party members. *Combat is one sided, set for you to loose. Even on the easy adventures. *Apparently all my adventures are stupid. Random bad luck occurs on every mission causing needless loss of life or permenantly bad statistics added to my characters. (yeah that is fun, instead of character progression, every quest my adventurers suck more) There are more reasons not to play this game but I am too frustrated to list them all. I would give this game another try if they update content to make it fun and rewarding.",
            "Redemption: Eternal Quest is an adventuerer management sim less about the adventures and more about the characters your send on those adventures. The game starts you out with a varaiaty of adventurers each randomly generated. You can have 6 in your guild at any time. Each character has his own stats and special trait that givem them an edge on adventures. The stats also affect things like how long a character will contract themselves to you and how much gold they want for doing so. For example, a character with high loyalty might be very willing to resign your pact for a few more years but one with low loyalty might rip you off for only a one year pact at a time. There are classes you can send characters on, three generated randomly at a time that last for what I think is a random number of seasons before changing to another class. Some classes take longer to train in than others. The adventures themselves are slightly interactive. You some times get choices like choosing to go left or right, what spot to explore in the dungeons (It should be noted that dungeons are all lyed out exactly the same with 3 3 branching choices with hidden outcomes placed randomly the layout of the dungeon has no real outcome on this) or wether or not to explore ruins. I found the flavor text of the adventures and the day to day guild life of the game to be varied enough to be interesting. I love this game, it's not a tripple A must have block buster, but for the price it is fun enough to waste a few hours on. You can play with various different game lengths and game modes to make the game easier or more difficult. My favorite so far is a short 25 year campaign with perminent death. It's a great game to play while on the phone with a friend or listening to music.",
            "Very interesting game. If you like management games, I would definitely recommend this. The dungeon crawl and minion mechanics are very unique. Good stuff!",
            "Redemption: Eternal Quest is a text based game with a team of warriors that fight and adventure on behalf of the player in a fantasy management game, the player can have up to six warriors at a time on the roster and send up to four of them on each adventure to find hidden treasure, ancient places of power, monsters to fight and many more interesting random events which can be both good or bad. Each warrior will have a pact in place much like a mercenary contract stating how much pay they require to work for you and how for how many years that service will be,the fees can vary from one time payments to yearly salary or even a percentage of the money made from treasure that's found along the journey but when a pact ends they might demand a better deal so this part of the game is a lot about balancing the books and deciding whether to have warriors fired or hire more which can be found at the hireling pool, at the end of the day this game is about staying a float with gold so tough decisions might have to be made. Each warrior also as different stats, traits, strengths and weakness that can help or hinder you through out your adventures, two in particular are combat and dexteritywhich form a score towards the battle card games that use many monster and magic based picture cards with a random score to determin a total battle score, you can use between one to three cards per battle if you have them and the total score must be higher than the enemies score to beat them, up to eight battle cardscan be held at any one time and you can earn them through battles, opening chests and buying them every season. If you are looking for a management game that isn't about Soccer then give this a try because it's a nice adventure game that uses the power of your imagination."
        ]
}

response = requests.post(url, json=payload)
print(response.json())

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
127.0.0.1 - - [22/Nov/2026 11:47:22] "POST /inference HTTP/1.1" 200 -


{'app_name': 'Redemption: Eternal Quest', 'aspects': [{'aspect': 'music', 'category': 'Audio', 'sentiment': 'Positive'}, {'aspect': 'interface', 'category': 'UI/UX', 'sentiment': 'Positive'}, {'aspect': 'characters', 'category': 'Gameplay', 'sentiment': 'Negative'}, {'aspect': 'combat', 'category': 'Gameplay', 'sentiment': 'Negative'}, {'aspect': 'campaign', 'category': 'Gameplay', 'sentiment': 'Negative'}, {'aspect': 'mechanics', 'category': 'Gameplay', 'sentiment': 'Neutral'}], 'raw_prediction': ['Aspects: [Audio:music:positive, UI/UX:interface:positive]', 'Aspects: [Gameplay:characters:negative, Gameplay:combat:negative]', 'Aspects: [Audio:music:positive, Gameplay:campaign:negative]', 'Aspects: [Gameplay:mechanics:neutral]', 'Aspects: [Gameplay:combat:negative]'], 'reviews': ["The interface needs a bit more polishing, and i was pretty confused at the initial team set up, I think a tutorial would be more friendly. Took me a while to really figure out what was going on, then i started

In [10]:
# server_thread.shutdown()
# server_thread.join()
# ngrok.kill()